In [28]:
import os
from dotenv import load_dotenv
load_dotenv()
from util import split_wav_by_timestamps

token = os.environ.get('HUGGINGFACE_ACCESS_TOKEN')
audio_file = "sample.wav"

In [ ]:
import whisperx

device = "cuda" 
batch_size = 16 # reduce if low on GPU mem
compute_type = "float16" # change to "int8" if low on GPU mem (may reduce accuracy)

# save model to local path (optional)
model_dir = "./data"
model = whisperx.load_model("large-v2", device, compute_type=compute_type, download_root=model_dir, language='en')

In [ ]:
audio = whisperx.load_audio(audio_file)
result = model.transcribe(audio, batch_size=batch_size)

In [ ]:
# 2. Align whisper output
model_a, metadata = whisperx.load_align_model(language_code=result["language"], device=device)
result = whisperx.align(result["segments"], model_a, metadata, audio, device, return_char_alignments=False)

In [ ]:
# 3. Assign speaker labels
diarize_model = whisperx.DiarizationPipeline(use_auth_token=token, device=device)

# add min/max number of speakers if known
diarize_segments = diarize_model(audio, min_speakers=2, max_speakers=2)

result = whisperx.assign_word_speakers(diarize_segments, result)

In [ ]:
timestamps = []
for segment in result["segments"]:
    if 'speaker' in segment.keys():
        speaker = segment['speaker']
    else:
        speaker = 'unknown'
    timestamps.append((segment["start"], segment["end"], speaker))

In [24]:
output_dir = 'out'

if not os.path.exists(output_dir):
    os.makedirs(output_dir)

split_wav_by_timestamps(audio_file, output_dir, timestamps)